# Impute expression with DIODE

Use the bundled data in `data/data_example/` to tune DIODE, identify outliers, and return a completed expression matrix.

## Instructions

The expression matrix has **genes as rows** and **cells as columns**. The gene file labels rows in order. The pseudotime file gives one value per cell in the same order as the columns.

| Gene | Cell 1 | Cell 2 | Cell 3 | … |
| --- | ---: | ---: | ---: | ---: |
| Gene1 | 0.6002 | 0.5964 | 0.5954 | … |
| Gene2 | 1.1481 | 1.1472 | 1.1025 | … |
| Gene3 | 1.0686 | 1.0412 | 1.0346 | … |
| … | … | … | … | … |

These are rounded values from the bundled dataset. The pseudotimes for the first three columns are `0.000000`, `0.020408`, and `0.040816`.

To use your own data, set `expression`, `genes`, and `pseudotime` in `configs/data_example.yaml`. Use a whitespace-delimited numeric matrix without a header, one gene name per line, and one pseudotime value per line.

In [2]:
from pathlib import Path
import yaml
from diode import DIODE

repo_root = Path.cwd()
if repo_root.name == "Example":
    repo_root = repo_root.parent
config_path = repo_root / "configs/data_example.yaml"

## Set the hyperparameter grid

Edit these candidate lists to tune DIODE. This cell writes your grid to `configs/data_example.yaml`, which also contains the training defaults. The other tuning tutorial reads and writes the same config.

In [3]:
# Define the hyperparameter grid for tuning.
hyperparameter_grid = {
    "huber_delta": [0.01, 0.05],
    "l2_lambda": [1e-07, 1e-05],
}

# To write the hyperparameter grid to the config file.
config = yaml.safe_load(config_path.read_text(encoding="utf-8"))
config["solvers"]["diode"]["hyperparameter_grid"] = hyperparameter_grid
with config_path.open("w", encoding="utf-8") as handle:
    yaml.safe_dump(config, handle, sort_keys=False, default_flow_style=None)

## Impute the expression matrix

DIODE tunes the grid and fits an ODE trajectory using the supplied pseudotimes. It flags an entry when the absolute difference between its observed and predicted values exceeds the selected Huber threshold. 

For each flagged entry, DIODE interpolates between the closest unflagged values for that gene on either side in pseudotime. The result is a genes-by-cells matrix in the original file order.

To choose which entries to impute, pass a Boolean mask as model.impute(mask=mask). Match the expression matrix's shape and order, with genes as rows and cells as columns. True replaces an entry regardless of its residual, and False keeps its original value, including zero. With mask=None or no mask argument, DIODE selects outliers using the tuned Huber threshold.

In [4]:
model = DIODE(config_path)
imputed_expression = model.impute()
imputed_expression

array([[ 0.60020594,  0.59644599,  0.59538989,  0.54698333,  0.52242619,
         0.54933016,  0.48059541,  0.43901057,  0.43685762,  0.43470468,
         0.38707891,  0.37330333,  0.35952775,  0.34987976,  0.34023177,
         0.33594437,  0.31712868,  0.298313  ,  0.27949731,  0.30068349,
         0.25185581,  0.27833907,  0.24592585,  0.23834417,  0.23076249,
         0.22318081,  0.19704036,  0.19609241,  0.21012549,  0.19621174,
         0.17756541,  0.15891908,  0.17624936,  0.17723378,  0.18198741,
         0.16980965,  0.17628224,  0.15945265,  0.16033194,  0.15405394,
         0.16266055,  0.13732381,  0.18806864,  0.16666478,  0.15950932,
         0.17644936,  0.17045407,  0.16398492,  0.15778884,  0.15159276],
       [ 1.14806443,  1.14723308,  1.10254701,  1.14231005,  1.13469734,
         1.10859162,  1.10093013,  1.11236427,  1.12379841,  1.09100115,
         1.09746439,  1.08669777,  1.09511786,  1.0876026 ,  1.08008733,
         1.07257207,  1.0683696 ,  1.06416712,  1.